# COGS 188 — Project Proposal
## University Course Timetabling with Search and Evolutionary Optimization

**Group members:** Derrick Chen, Zhen Wang, and Daira Merino  
**Project type:** Implementation  
**Repository:** [zhc066/COGS188_group](https://github.com/zhc066/COGS188_group)

# Abstract

We propose to compare AI search methods for constructing university course timetables. The task is to assign weekly lecture meetings to rooms and time slots while avoiding room, instructor, and student-cohort conflicts. Among feasible schedules, we will minimize penalties for inconvenient gaps, undesirable meeting times, and changes of room within a course. We will generate reproducible scheduling instances with different sizes and levels of resource contention. A greedy scheduler will provide a baseline, and we will implement simulated annealing and a genetic algorithm to investigate whether broader exploration improves schedule quality. We will evaluate feasibility rate, preference penalties, runtime, and consistency across random seeds under equal computation budgets. Our central question is how the methods' tradeoffs change as scheduling becomes more constrained. The intended outcome is a reproducible comparison and timetable visualization, rather than a production scheduling system.

# Background

University timetabling combines resource allocation with conflicting requirements. The ITC-2007 curriculum-based formulation assigns lectures to periods and rooms, with conflicts determined by instructors and curricula [1]. We will use this as background for a simplified, explicitly documented formulation; our scores will not be presented as official ITC benchmark scores.

Local search represents a candidate timetable as a complete assignment and improves it through small changes. Simulated annealing allows some unfavorable moves, reducing their probability as temperature falls [2, 3]. Genetic algorithms instead search through populations using selection, crossover, and mutation [2]. These approaches offer a useful comparison between improving one candidate and maintaining multiple candidates. Our hypothesis is that both will improve preference satisfaction over a single greedy pass, but that the genetic algorithm's population overhead may make it less competitive under short time limits. This is a hypothesis to test, not an assumed result.

# Problem Statement

**Research question:** Under a fixed computation budget, which method most reliably produces feasible, low-penalty course timetables as resource contention increases?

An instance contains courses, required weekly meetings, instructors, student cohorts, rooms, and available periods. Each meeting lasts one period. Each course has one instructor and may belong to multiple cohorts. A schedule assigns every required meeting to exactly one room and period.

**Hard constraints**
- A room cannot host two meetings in the same period.
- An instructor cannot teach two meetings in the same period.
- Courses sharing a cohort cannot meet simultaneously.
- Meetings must respect instructor availability and room capacity.
- Multiple meetings of a course must occur on different days.

**Soft constraints**
- Reduce idle periods between a cohort's first and last meeting of each day.
- Avoid designated undesirable periods.
- Keep each course in as few different rooms as possible.

Let H be the total hard-constraint violation count. Define S = G + U + R, where G counts cohort idle periods, U counts meetings in undesirable periods, and R sums max(0, distinct rooms used − 1) across courses. We will document exact counting conventions and keep these unit weights fixed in the primary comparison.

Feasibility takes priority over preferences. Search will minimize E = M H + S, with M greater than a computed upper bound on S for each instance. Thus, reducing one hard violation always outweighs any possible preference improvement. Final results will distinguish feasible and infeasible outputs.

The problem is non-trivial because assignments interact across several shared resources. For L meetings, P periods, and R rooms, the unconstrained assignment space contains (P R)^L possibilities; straightforward enumeration quickly becomes impractical.

# Data

Our primary inputs will be **synthetic scheduling instances**, not private university records. An observation is an entire scheduling instance, with linked course, meeting, room, and availability tables.

We plan to generate 1,200 instances: 200 for development and parameter selection and 1,000 held out for evaluation. Proposed instance sizes are 20, 40, and 60 courses, 5–10 rooms, and five teaching days with six periods per day. Each course will require two or three weekly meetings. We will vary cohort overlap, instructor availability, and room demand.

Recorded fields will include instance ID, generation seed, course ID, instructor ID, cohort IDs, enrollment, required meeting count, meeting ID, room ID, room capacity, day, period, availability, and undesirable-period indicators. Lists and availability masks will be encoded explicitly in JSON; generated configurations and seeds will be versioned.

To avoid unintentionally impossible primary cases, we will first construct a valid hidden timetable and derive compatible availability, room capacities, and conflicts from it. Solvers will receive only the resulting instance. The hidden timetable will verify feasibility, not serve as an optimum or solver initialization. Separate seeds will produce development and test instances.

We will validate identifiers, references, required meetings, capacities, and the hidden timetable with an independent constraint checker. We will report distributions of instance size and resource contention. Because this is an optimization project rather than supervised prediction, the instances are search problems rather than labeled training examples. We request approval for this formulation and its synthetic evaluation design; synthetic realism remains a limitation.

# Proposed Solution

We will implement a shared Python representation, objective function, and independent schedule validator, then compare:

1. **Greedy baseline.** Order meetings by the number of conflicts and available assignments, placing the most constrained first. Choose a feasible placement with the smallest added preference penalty. If no feasible placement exists, use the placement with the smallest added E and record the resulting violations. Use fixed tie-breaking.
2. **Simulated annealing.** Begin from the greedy timetable. Propose moving one meeting or swapping two meetings. Accept improvements and accept a worsening change of size ΔE with probability exp(−ΔE/T). Gradually decrease T and retain the best candidate encountered.
3. **Genetic algorithm.** Encode one room-period assignment per meeting. Initialize a population containing the greedy timetable and randomized candidates. Use tournament selection, course-level crossover, meeting reassignment mutation, and elitism. Apply a bounded conflict-repair procedure after crossover and mutation. Repair time counts toward the runtime budget.

Our main implementation will use Python, NumPy, pandas, and Matplotlib, with pinned dependencies and recorded seeds. We will request instructor approval for any methods outside the course coverage.

**Parameter selection:** Compare a small, equal-budget set of development configurations: annealing temperatures and cooling rates, and genetic population sizes and mutation rates. Select settings using feasibility first and preference penalty second, then freeze them before evaluating held-out instances. The score weights will not be tuned to favor a method.

**Scope:** The required deliverable is the three-method comparison on our documented formulation. An optional extension is to parse a small selection of public timetabling instances after matching their constraints; it is not required for the core project. At proposal time, this repository contains the plan and report template; no working scheduler or experimental results are claimed.

# Evaluation Metrics

All methods will use the same instances, validator, and score. We will report:

- **Feasibility rate:** number of runs with H = 0 divided by total runs.
- **Preference penalty:** S and S/L for feasible outputs, plus the individual G, U, and R components. Failed runs remain visible and are not treated as low-penalty successes.
- **Computation:** elapsed time to the first feasible timetable, objective evaluations, and best feasible penalty at fixed time budgets. Runs that never become feasible will be recorded as failures at the cutoff.
- **Scalability and variability:** results by course count and resource contention, with median and interquartile range across instances and seeds.

The initial evaluation will allow one second per run, including initialization and repair, on one documented machine. Each stochastic method will use three seeds per test instance; the deterministic greedy baseline will run once per instance. A preselected stratified subset will also receive five-second runs to examine improvement with additional time. Development timing will determine whether budgets need adjustment before test evaluation.

We will compare methods on matched instances and show feasibility curves, penalty distributions, and example timetable grids. Quality comparisons will also use the subset of instances where all methods find feasible outputs, alongside each method's full feasibility rate, to make differing success rates clear. We will report paired penalty differences and avoid claiming optimality without a proven bound.

# Ethics & Privacy

We will use synthetic instructors and cohorts and collect no personal schedules or student records. Preference weights can favor some groups, and an aggregate score may conceal one cohort receiving a particularly inconvenient timetable. We will therefore also inspect per-cohort idle time and undesirable-period assignments. Our simplified constraints omit factors such as accessibility, travel time, and individual accommodations; the resulting timetables will be research demonstrations, not recommendations for actual course allocation.

# Team Expectations

The following is a proposed working agreement for the group to review:
- Meet weekly, record decisions in the repository, and flag blockers early.
- Share implementation, experiments, and writing; each member reviews another member's work.
- Agree on the data schema, constraint definitions, and evaluation protocol before implementing separate methods.
- Use development evidence to resolve technical disagreements and document changes to scope.
- Set an internal report deadline at least two days before the official deadline.

**Proposed responsibilities:** Derrick Chen will lead the instance generator, validator, and greedy baseline; Zhen Wang will lead simulated annealing; Daira Merino will lead the genetic algorithm. All three will integrate the code, run experiments, interpret results, and write the report. These assignments can be adjusted by the team.

# Project Timeline Proposal

Dates will be aligned with the course schedule; the phases below are relative to proposal approval.

| Phase | Planned work | Lead |
|---|---|---|
| Week 1 | Finalize constraints; generate sample instances; implement validator and greedy baseline | Derrick; all review |
| Week 2 | Implement annealing and genetic methods; verify outputs on small cases | Zhen and Daira |
| Week 3 | Tune on development instances; freeze parameters and evaluation protocol | All |
| Week 4 | Run held-out comparisons; analyze failures, runtime, and schedule quality | All |
| Before finals-week deadline | Complete report, figures, and reproduction instructions; review and export final PDF | All |

The final report will include background and citations, methods, results, discussion, and conclusion, with a link to this public repository containing source code, generation scripts, configurations, and experiment outputs. Submission will follow the official Gradescope deadline.

# References

[1] Müller, T. (2008). *ITC2007 Solver Description: A Hybrid Approach.* [Author's paper](https://muller.unitime.org/itc2007.pdf). Background on curriculum-based timetabling and its constraints.

[2] Russell, S., and Norvig, P. *Artificial Intelligence: A Modern Approach — Local Search Algorithms*, Chapter 4 lecture slides. [Authors' slides](https://aima.eecs.berkeley.edu/slides-pdf/chapter04b-6pp.pdf). Local search, simulated annealing, and genetic algorithms.

[3] Kirkpatrick, S., Gelatt, C. D., Jr., and Vecchi, M. P. (1983). Optimization by Simulated Annealing. *Science*, 220(4598), 671–680. [https://doi.org/10.1126/science.220.4598.671](https://doi.org/10.1126/science.220.4598.671).
